# Actividad 3: Aplicación de algoritmos de aprendizaje supervisado con PySpark

**Materia:** Análisis de grandes volúmenes de datos  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Autor:** Jonathan Javier Monsalve Giraldo (A01840272)  
**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 24 de mayo de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025  
**Modalidad:** Individual

## Objetivo

Aplicar un algoritmo de aprendizaje supervisado en PySpark MLlib sobre una muestra M' derivada de la muestra estratificada M construida en la Etapa 2 del proyecto del equipo. El problema elegido es regresión sobre `fare_amount` (tarifa base registrada del viaje), enmarcado como un modelo operativo de auditoría tarifaria a partir de variables del registro del viaje.

## Estructura del notebook

1. **Introducción**: aprendizaje supervisado, algoritmos representativos y los disponibles en PySpark MLlib.
2. **Selección de los datos**: reconstrucción compacta de M (recap de la Etapa 2) y construcción de la muestra individual M'.
3. **Preparación del conjunto de entrenamiento y prueba**: partición estratificada train/test y validación del split.
4. **Construcción de modelos de aprendizaje**: definición del problema, features, pipeline, modelos y resultados.

### Nota para el profesor

La Sección 2.0 reproduce de forma compacta la muestra M de la Etapa 2. El aporte individual inicia en la **Sección 2.1**, con la construcción de M', la partición train/test y el modelado supervisado.

## 1. Introducción

### 1.1 Aprendizaje supervisado

El aprendizaje supervisado es el paradigma en el que un modelo aprende una relación entre un conjunto de variables predictoras y una variable objetivo conocida en los datos de entrenamiento, para después aplicar esa relación a observaciones nuevas. Cada fila del conjunto de entrenamiento incluye tanto las predictoras como la respuesta etiquetada, y la calidad del modelo se evalúa sobre datos no usados durante el ajuste para estimar su capacidad de generalización.

Se distinguen dos grandes tareas según la naturaleza de la variable objetivo. En **regresión** la respuesta es continua, como el monto de una tarifa, la demanda esperada o la temperatura. En **clasificación** la respuesta es categórica, binaria o multiclase, como fraude vs no fraude, especie de una flor o tipo de pago. Esta actividad aborda un problema de regresión.

### 1.2 Algoritmos representativos

La literatura agrupa los algoritmos supervisados en familias con supuestos y compromisos distintos:

- **Modelos lineales** (regresión lineal, regresión logística): coeficientes interpretables, supuesto de linealidad, sensibles a multicolinealidad y a la escala de los predictores. Útiles como baseline y cuando la relación esperada es aproximadamente lineal.
- **Árboles de decisión**: particiones recursivas del espacio de features, capturan no linealidades e interacciones, fáciles de interpretar como reglas; un árbol único tiende a sobreajustar si la profundidad crece.
- **Ensembles de árboles** (Random Forest, Gradient Boosted Trees): combinan muchos árboles para reducir varianza o sesgo. Suelen rendir bien en datos tabulares y exponen importancia de variables, a costa de menor interpretabilidad y mayor costo de cómputo.
- **Máquinas de vectores de soporte (SVM)**: separan clases maximizando el margen, eficaces con datos de alta dimensión y kernels para relaciones no lineales; menos comunes en Big Data por su costo cuadrático.
- **Perceptrón multicapa (MLP)**: redes feedforward capaces de aprender relaciones complejas; requieren más datos, tuning cuidadoso y aportan poca interpretabilidad directa.
- **Naive Bayes**: clasificador probabilístico basado en independencia condicional entre features; rápido y útil en problemas de conteo o texto.

### 1.3 Disponibles en PySpark MLlib

PySpark expone los algoritmos anteriores a través del módulo moderno `pyspark.ml`, basado en DataFrames y organizado bajo el patrón Estimator-Transformer-Pipeline. Un *Estimator* aprende parámetros con `fit()` (por ejemplo, `LinearRegression`, `RandomForestClassifier`); un *Transformer* aplica una transformación con `transform()` (por ejemplo, un modelo ya entrenado o un `VectorAssembler`); un *Pipeline* encadena pasos para garantizar que el mismo preprocesamiento aprendido en train se aplique a test, evitando fuga de información.

| Tarea | Submódulo | Algoritmos disponibles |
|---|---|---|
| Regresión | `pyspark.ml.regression` | `LinearRegression`, `GeneralizedLinearRegression`, `DecisionTreeRegressor`, `RandomForestRegressor`, `GBTRegressor`, `IsotonicRegression`, `AFTSurvivalRegression`, `FMRegressor` |
| Clasificación | `pyspark.ml.classification` | `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`, `GBTClassifier`, `MultilayerPerceptronClassifier`, `NaiveBayes`, `LinearSVC`, `OneVsRest`, `FMClassifier` |
| Evaluación | `pyspark.ml.evaluation` | `RegressionEvaluator`, `BinaryClassificationEvaluator`, `MulticlassClassificationEvaluator` |
| Tuning | `pyspark.ml.tuning` | `ParamGridBuilder`, `CrossValidator`, `TrainValidationSplit` |

Esta actividad utiliza `LinearRegression` como baseline interpretable y `RandomForestRegressor` como modelo principal, con `RegressionEvaluator` para las métricas y un `Pipeline` para el preprocesamiento.

### 1.4 Referencias

Apache Software Foundation. (2026). *MLlib (DataFrame-based): PySpark 4.1.2 documentation*. https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html

Apache Software Foundation. (2026). *RegressionEvaluator: PySpark 4.1.2 documentation*. https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html

Géron, A. (2022). *Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow: Concepts, tools, and techniques to build intelligent systems* (3rd ed.). O'Reilly Media.

Polak, A. (2023). *Scaling machine learning with Spark: Distributed ML with MLlib, TensorFlow, and PyTorch*. O'Reilly Media.

## 2. Selección de los datos

### 2.0 Reconstrucción compacta de la muestra M (recap de Etapa 2)

> **Nota al profesor:** las próximas cinco celdas reproducen el muestreo estratificado de la **Etapa 2** del proyecto (downcast del esquema, filtros destructivos, imputaciones con auditoría de nulos, construcción del `stratum_id` y `sampleBy` con piso por estrato). Si está familiarizado con esa entrega, puede saltar directamente a la **Sección 2.1**, donde construyo la muestra individual M' a partir de M con ventana exacta. La 2.0 está aquí para que el notebook sea autocontenido y reproducible en Colab.

El bloque se ejecuta en 5 celdas: (a) setup de Spark y rutas; (b) descarga idempotente de los 24 parquets, downcast del esquema y filtros destructivos; (c) imputaciones de seis columnas y auditoría de nulos; (d) construcción del estrato y derivación de `stratum_id`; (e) recálculo del diccionario de fracciones desde primeros principios y extracción de M vía `sampleBy`.

**Instalación de dependencias.** La celda siguiente instala las librerías necesarias antes del bloque.

In [1]:
# Dependencias de Python para el notebook. Idempotente.
!pip install -q pyspark findspark pandas

# Solo en Google Colab: descomentar para instalar Java (JVM de Spark).
# Localmente con env-pyspark esta línea no es necesaria.
# !apt-get install openjdk-8-jdk-headless -qq > /dev/null



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
# (a) setup de Spark y rutas
import findspark
findspark.init()

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pathlib import Path
import urllib.request

spark = (SparkSession.builder
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.debug.maxToStringFields", 100)
    .getOrCreate())

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 19:21:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1


In [3]:
# (b) descarga idempotente de los 24 parquets, downcast del esquema y filtros destructivos
CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
YEARS = (2024, 2025)

def fetch(url, target):
    if target.exists():
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, target)

for y in YEARS:
    for m in range(1, 13):
        f = f"yellow_tripdata_{y}-{m:02d}.parquet"
        fetch(f"{CDN_BASE}/trip-data/{f}", DATA_DIR / f)
fetch(f"{CDN_BASE}/misc/taxi_zone_lookup.csv", DATA_DIR / "taxi_zone_lookup.csv")

df_native = (spark.read.option("mergeSchema", "true")
    .parquet(*sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))))
zones = (spark.read.option("header", True).option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv")))

df_raw = df_native.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

df_filtered = (df_raw
    .filter(F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    .filter(F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
    .filter(F.col("trip_distance").between(0, 200))
    .filter(F.col("fare_amount").between(0, 1000))
    .filter(F.col("total_amount").between(0, 1200))
    .filter(~((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0))))

n_raw, n_filtered = df_raw.count(), df_filtered.count()
print(f"Crudo: {n_raw:,} | Tras filtros: {n_filtered:,} | Perdida: {(n_raw - n_filtered) / n_raw * 100:.2f}%")
assert (n_raw - n_filtered) / n_raw < 0.15

Crudo: 89,892,322 | Tras filtros: 84,437,138 | Perdida: 6.07%


In [4]:
# (c) imputaciones de seis columnas y auditoría de nulos
df_clean = (df_filtered
    .withColumn("passenger_count",
        F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))
         .otherwise(F.lit(1).cast("byte")))
    .withColumn("cbd_congestion_fee",
        F.when(F.col("cbd_congestion_fee").isNull() | (F.col("tpep_pickup_datetime") < F.lit("2025-01-05")),
               F.lit(0.0).cast("float"))
         .otherwise(F.col("cbd_congestion_fee")))
    .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge"), F.lit(0.0).cast("float")))
    .withColumn("Airport_fee", F.coalesce(F.col("Airport_fee"), F.lit(0.0).cast("float")))
    .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), F.lit(99).cast("byte")))
    .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), F.lit("F"))))

imputed_cols = ["passenger_count", "cbd_congestion_fee", "congestion_surcharge",
                "Airport_fee", "RatecodeID", "store_and_fwd_flag"]
nulls = df_clean.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in imputed_cols]).first()
assert all((nulls[c] or 0) == 0 for c in imputed_cols), f"Nulos remanentes: {nulls.asDict()}"
print("Imputaciones aplicadas; 0 nulos en las 6 columnas objetivo.")

Imputaciones aplicadas; 0 nulos en las 6 columnas objetivo.


In [5]:
# (d) construcción del estrato y derivación de `stratum_id`
airport_ids = {r.LocationID for r in zones.filter(F.col("service_zone").isin("Airports", "EWR")).collect()}
unknown_ids = {264, 265}
manhattan_ids = {r.LocationID for r in zones.filter(F.col("Borough") == "Manhattan").collect()} - airport_ids - unknown_ids
outer_ids = {r.LocationID for r in zones.filter(F.col("Borough").isin("Brooklyn", "Queens", "Bronx", "Staten Island")).collect()} - airport_ids - unknown_ids

df_feat = (df_clean
    .withColumn("pu_macro_zone",
        F.when(F.col("PULocationID").isin(sorted(airport_ids)), "airport")
         .when(F.col("PULocationID").isin(sorted(unknown_ids)), "unknown")
         .when(F.col("PULocationID").isin(sorted(manhattan_ids)), "manhattan")
         .when(F.col("PULocationID").isin(sorted(outer_ids)), "outer_borough")
         .otherwise("unknown"))
    .withColumn("payment_group",
        F.when(F.col("payment_type") == 0, "flex")
         .when(F.col("payment_type") == 1, "credit")
         .when(F.col("payment_type") == 2, "cash")
         .otherwise("other"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("day_hour_bucket",
        F.when(F.col("pickup_hour").between(0, 5), "late_night")
         .when(F.col("dow").isin(1, 7), "weekend")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(6, 10), "weekday_am")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(16, 20), "weekday_pm_peak")
         .otherwise("other"))
    .withColumn("trip_distance_bin",
        F.when(F.col("trip_distance") < 1.12, "short")
         .when(F.col("trip_distance") < 12.43, "medium")
         .otherwise("long"))
    .withColumn("is_flex_fare", F.col("payment_type") == 0)
    .withColumn("cbd_period_flag",
        F.when(F.col("tpep_pickup_datetime") < F.lit("2025-01-05"), "pre_cbd").otherwise("post_cbd"))
    .withColumn("stratum_id",
        F.concat_ws("|",
            F.col("pu_macro_zone"), F.col("payment_group"),
            F.col("day_hour_bucket"), F.col("trip_distance_bin"))))
print("Variables de estrato construidas:", sorted(set(df_feat.columns) - set(df_clean.columns)))

Variables de estrato construidas: ['cbd_period_flag', 'day_hour_bucket', 'dow', 'is_flex_fare', 'payment_group', 'pickup_hour', 'pu_macro_zone', 'stratum_id', 'trip_distance_bin']


In [6]:
# (e) recálculo del diccionario de fracciones desde primeros principios y extracción de M vía `sampleBy`
ESTIMATED_M = 5_030_141
N_M_TARGET = 5_000_000
MIN_FLOOR_M = 500

strata_D = (df_feat.groupBy("stratum_id").count()
    .withColumnRenamed("count", "n_D")
    .withColumn("target_n",
        F.least(F.col("n_D"),
                F.greatest(F.lit(MIN_FLOOR_M).cast("long"),
                           F.round(F.lit(N_M_TARGET) * F.col("n_D") / F.lit(n_filtered)).cast("long"))))
    .withColumn("fraction", F.col("target_n") / F.col("n_D")))

fractions = {r["stratum_id"]: float(r["fraction"]) for r in strata_D.select("stratum_id", "fraction").collect()}
assert all(0 < f <= 1.0 for f in fractions.values())

M = df_feat.stat.sampleBy("stratum_id", fractions, seed=42).cache()
n_M = M.count()
print(f"|M| = {n_M:,} (objetivo {N_M_TARGET:,}, esperado ~5.03M)")
assert abs(n_M - ESTIMATED_M) / ESTIMATED_M < 0.02, f"|M| diverge: {n_M:,}"

# Fin de reconstrucción Etapa 1 y 2
M.select("stratum_id", "fare_amount", "trip_distance", "pu_macro_zone", "payment_group").show(5, truncate=False)

|M| = 5,029,725 (objetivo 5,000,000, esperado ~5.03M)
+--------------------------------------+-----------+-------------+-------------+-------------+
|stratum_id                            |fare_amount|trip_distance|pu_macro_zone|payment_group|
+--------------------------------------+-----------+-------------+-------------+-------------+
|manhattan|credit|late_night|medium    |22.6       |5.72         |manhattan    |credit       |
|manhattan|credit|late_night|medium    |35.9       |7.2          |manhattan    |credit       |
|manhattan|credit|late_night|medium    |16.3       |3.67         |manhattan    |credit       |
|outer_borough|credit|late_night|medium|14.2       |2.67         |outer_borough|credit       |
|manhattan|credit|other|short          |8.6        |0.87         |manhattan    |credit       |
+--------------------------------------+-----------+-------------+-------------+-------------+
only showing top 5 rows


## 2.1 Construcción de M' a partir de M

A partir de M (5.03M filas) construimos M', una submuestra individual de tamaño manejable que preserva el diseño estratificado de Etapa 2. La técnica es **ventana exacta por estrato**:

- Para cada uno de los 240 estratos s (valores únicos de `stratum_id`), calculamos `target_n_s = max(50, floor(0.20 * n_M_s))`. El piso de 50 garantiza determinísticamente que un split 80/20 posterior (Sección 3) deje al menos 10 filas en test, incluso en los estratos más chicos heredados del piso de Etapa 2.
- Sobre `Window.partitionBy("stratum_id").orderBy(F.rand(seed=42))` asignamos `row_number()` y filtramos `rn <= target_n_s`. Esto materializa exactamente `target_n_s` filas por estrato.

El método es simétrico al split train/test de la Sección 3 (mismo patrón de ventana) y consistente con la inversión estratificada de Etapa 2.

In [7]:
# construcción de M' por ventana exacta con piso determinístico de 50
F_GLOBAL_MP = 0.20
MIN_FLOOR_MP = 50

counts_M = M.groupBy("stratum_id").count().withColumnRenamed("count", "n_M_s")
target_Mp = counts_M.withColumn(
    "target_n_s",
    F.least(
        F.col("n_M_s"),
        F.greatest(F.lit(MIN_FLOOR_MP).cast("long"),
                   F.floor(F.lit(F_GLOBAL_MP) * F.col("n_M_s")).cast("long"))))

w_Mp = Window.partitionBy("stratum_id").orderBy(F.rand(seed=42))
M_prime = (M.join(target_Mp, "stratum_id")
    .withColumn("rn", F.row_number().over(w_Mp))
    .filter(F.col("rn") <= F.col("target_n_s"))
    .drop("rn", "target_n_s", "n_M_s")
    .cache())

n_Mp = M_prime.count()
print(f"|M'| = {n_Mp:,} (esperado ~1.0M)")
assert 950_000 <= n_Mp <= 1_100_000, f"|M'| fuera de rango: {n_Mp:,}"

|M'| = 1,006,344 (esperado ~1.0M)


## 2.2 Validación de representatividad M' vs M

Tres validaciones obligatorias para verificar que M' preserva la estructura estratificada de M:

1. **Tamaños y piso**: |M|, |M'|, y conteo mínimo por estrato en M' (debe ser >= 50 por el piso determinístico).
2. **Cardinalidad de estratos**: M' debe contener los 240 estratos de M.
3. **Marginales** en las cuatro variables del estrato (`pu_macro_zone`, `payment_group`, `day_hour_bucket`, `trip_distance_bin`): desviación máxima entre p_M y p_M' por categoría.

In [8]:
# validación compacta M' vs M
n_M_v, n_Mp_v = M.count(), M_prime.count()
strata_M = M.select("stratum_id").distinct().count()
strata_Mp = M_prime.select("stratum_id").distinct().count()
min_count_Mp = M_prime.groupBy("stratum_id").count().agg(F.min("count")).first()[0]

print(f"|M| = {n_M_v:,}  |M'| = {n_Mp_v:,}  ratio = {n_Mp_v / n_M_v:.4f}")
print(f"Estratos en M = {strata_M}, en M' = {strata_Mp} (debe ser 240)")
print(f"Piso mínimo por estrato en M' = {min_count_Mp} (debe ser >= 50)\n")
assert strata_Mp == strata_M, "M' perdió estratos"
assert min_count_Mp >= 50, f"Piso violado: {min_count_Mp}"

# Marginales en las 4 variables del estrato
for col in ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]:
    p_M = {r[col]: r["count"] / n_M_v for r in M.groupBy(col).count().collect()}
    p_Mp = {r[col]: r["count"] / n_Mp_v for r in M_prime.groupBy(col).count().collect()}
    max_diff_pp = max(abs(p_M.get(k, 0) - p_Mp.get(k, 0)) for k in set(p_M) | set(p_Mp)) * 100
    print(f"  {col}: max |p_M - p_M'| = {max_diff_pp:.4f} pp")
    assert max_diff_pp < 0.5, f"Marginal de {col} diverge: {max_diff_pp:.4f} pp"

|M| = 5,029,725  |M'| = 1,006,344  ratio = 0.2001
Estratos en M = 240, en M' = 240 (debe ser 240)
Piso mínimo por estrato en M' = 50 (debe ser >= 50)

  pu_macro_zone: max |p_M - p_M'| = 0.0369 pp
  payment_group: max |p_M - p_M'| = 0.0307 pp
  day_hour_bucket: max |p_M - p_M'| = 0.0079 pp
  trip_distance_bin: max |p_M - p_M'| = 0.0287 pp


## 3. Preparación del conjunto de entrenamiento y prueba

**Justificación del 80/20**: la proporción se elige por su interacción exacta con el piso de 50 filas por estrato heredado de la sección 2.1. `floor(0.8 * 50) = 40` filas de train y `50 - 40 = 10` filas de test en los estratos más chicos: train suficiente para que el modelo vea el caso raro y test con el mínimo razonable para una estadística por estrato. Un split más agresivo (90/10) dejaría 5 filas en test, insuficiente; uno más conservador (70/30) restaría 5 filas de train sin ganancia útil dado que 10 ya es el piso adecuado. Sobre el total (M' ~1M filas), el conjunto de test de ~200k tiene precisión más que suficiente para estimar RMSE, MAE y R² con error estándar despreciable. El 80/20 también es la convención estándar para conjuntos medianos a grandes en aprendizaje supervisado.

**Técnica**: ventana exacta por estrato, mismo patrón usado en 2.1 para construir M': `Window.partitionBy("stratum_id").orderBy(F.rand(seed=123))` con `row_number()` y filtro `rn <= floor(0.8 * n_s)` para train, complemento para test. Esto materializa exactamente `floor(0.8 * n_Mp_s)` filas de train y `n_Mp_s - floor(0.8 * n_Mp_s)` de test por estrato. El seed difiere del usado en 2.1 (42) para mantener independencia metodológica entre las dos etapas de muestreo.

**¿Por qué no `randomSplit([0.8, 0.2])`?**: Bernoulli puro por fila, ignora la columna de estrato; deja varianza muestral que en estratos chicos (53 filas en el extremo de Etapa 2) puede romper la inversión estratificada.

**¿Por qué no `sampleBy("stratum_id", {sid: 0.8})`?**: también Bernoulli, ahora por estrato; no garantiza conteo exacto. La ventana exacta es el equivalente PySpark del `stratify=y` de scikit-learn.

In [9]:
# split estratificado exacto 80/20 sobre M_prime
TRAIN_RATIO = 0.8

counts_Mp = M_prime.groupBy("stratum_id").count().withColumnRenamed("count", "n_Mp_s")
w_split = Window.partitionBy("stratum_id").orderBy(F.rand(seed=123))

Mp_with_rn = (M_prime.join(counts_Mp, "stratum_id")
    .withColumn("rn", F.row_number().over(w_split))
    .withColumn("train_cutoff", F.floor(F.lit(TRAIN_RATIO) * F.col("n_Mp_s")).cast("long")))

train_df = (Mp_with_rn.filter(F.col("rn") <= F.col("train_cutoff"))
            .drop("rn", "train_cutoff", "n_Mp_s").cache())
test_df  = (Mp_with_rn.filter(F.col("rn") >  F.col("train_cutoff"))
            .drop("rn", "train_cutoff", "n_Mp_s").cache())

n_train, n_test = train_df.count(), test_df.count()
print(f"|train| = {n_train:,}  |test| = {n_test:,}  ratio_train = {n_train / (n_train + n_test):.4f}")

|train| = 804,991  |test| = 201,353  ratio_train = 0.7999


In [10]:
# verificación post-split: cardinalidad, piso y marginales train vs test
strata_train = train_df.select("stratum_id").distinct().count()
strata_test = test_df.select("stratum_id").distinct().count()
min_train = train_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]
min_test = test_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]

print(f"Estratos train = {strata_train}, test = {strata_test} (esperado 240)")
print(f"Piso min en train = {min_train}, en test = {min_test} (test debe ser >= 10)\n")
assert strata_train == 240 and strata_test == 240
assert min_test >= 10, f"Piso en test violado: {min_test}"

# Marginales train vs test en las 4 variables del estrato
for col in ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]:
    p_tr = {r[col]: r["count"] / n_train for r in train_df.groupBy(col).count().collect()}
    p_te = {r[col]: r["count"] / n_test for r in test_df.groupBy(col).count().collect()}
    max_diff_pp = max(abs(p_tr.get(k, 0) - p_te.get(k, 0)) for k in set(p_tr) | set(p_te)) * 100
    print(f"  {col}: max |p_train - p_test| = {max_diff_pp:.4f} pp")
    assert max_diff_pp < 0.5, f"Marginal de {col} diverge: {max_diff_pp:.4f} pp"

Estratos train = 240, test = 240 (esperado 240)
Piso min en train = 40, en test = 10 (test debe ser >= 10)

  pu_macro_zone: max |p_train - p_test| = 0.0315 pp
  payment_group: max |p_train - p_test| = 0.0215 pp
  day_hour_bucket: max |p_train - p_test| = 0.0066 pp
  trip_distance_bin: max |p_train - p_test| = 0.0176 pp


## 4. Construcción de modelos de aprendizaje

### 4.1 Definición del problema y variable objetivo

El problema supervisado es **regresión sobre `fare_amount`**, la tarifa base registrada del viaje. La motivación es auditoría tarifaria post-registro: dados los atributos operativos observados al cierre del viaje, estimar si la tarifa base es coherente.

Para evitar fuga de información se excluyen **nueve columnas**: `total_amount` y ocho componentes de cobro (`tip_amount`, `tolls_amount`, `extra`, `mta_tax`, `improvement_surcharge`, `congestion_surcharge`, `Airport_fee`, `cbd_congestion_fee`). `total_amount` contiene directamente a `fare_amount`; los demás componentes son cargos posteriores o paralelos al cálculo de la tarifa base.

### 4.2 Selección y derivación de features

Ocho predictores, todos disponibles en el registro del viaje:

| Feature | Tipo | Justificación |
|---|---|---|
| `trip_distance` | numérica continua | Distancia recorrida; principal componente físico de la tarifa |
| `trip_duration_min` | numérica continua | Duración real del viaje; aproxima tiempo de taxímetro y tráfico sin usar columnas de cobro |
| `pu_macro_zone` | categórica (4) | Captura origen aeroportuario y diferencias geográficas gruesas |
| `RatecodeID` | categórica (7) | Selector del régimen tarifario (JFK flat, Newark, Negotiated, Flex) |
| `is_flex_fare` | binaria | Marca régimen upfront pricing |
| `day_hour_bucket` | categórica (5) | Resume patrones temporales sin alta cardinalidad |
| `cbd_period_flag` | binaria | Marca el cambio regulatorio del CBD fee desde 2025-01-05 |
| `passenger_count` | numérica entera | Bajo costo; conserva información operativa básica |

`trip_duration_min` se deriva como diferencia entre `tpep_dropoff_datetime` y `tpep_pickup_datetime`. Se filtra a 0.5-720 minutos para remover duraciones imposibles o registros incompletos; la pérdida observada es menor a 1%.

In [11]:
# definición de target, features y verificación de ausencia de fuga
TARGET = "fare_amount"
CAT_COLS = ["pu_macro_zone", "RatecodeID", "day_hour_bucket", "cbd_period_flag"]
NUM_COLS = ["trip_distance", "trip_duration_min", "is_flex_fare", "passenger_count"]
FEATURE_COLS = [
    "trip_distance", "trip_duration_min", "pu_macro_zone", "RatecodeID", "is_flex_fare",
    "day_hour_bucket", "cbd_period_flag", "passenger_count",
]
LEAKAGE_COLS = {"total_amount", "tip_amount", "tolls_amount", "extra", "mta_tax",
                "improvement_surcharge", "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"}

assert set(FEATURE_COLS) == set(NUM_COLS + CAT_COLS), "FEATURE_COLS no coincide con NUM_COLS + CAT_COLS"
assert not (set(FEATURE_COLS) & LEAKAGE_COLS), "Hay fuga en FEATURE_COLS"
assert TARGET not in FEATURE_COLS, "El target no debe estar entre las features"

# Derivación de feature post-registro: duración del viaje en minutos.
def add_trip_duration(df):
    return (df
        .withColumn("trip_duration_min_raw",
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / F.lit(60.0))
        .filter(F.col("trip_duration_min_raw").between(0.5, 720.0))
        .withColumn("trip_duration_min", F.col("trip_duration_min_raw").cast("float"))
        .drop("trip_duration_min_raw"))

train_ml = (add_trip_duration(train_df)
    .withColumn("is_flex_fare", F.col("is_flex_fare").cast("byte"))
    .cache())
test_ml = (add_trip_duration(test_df)
    .withColumn("is_flex_fare", F.col("is_flex_fare").cast("byte"))
    .cache())

missing = sorted(set(FEATURE_COLS) - set(train_ml.columns))
assert not missing, f"Features ausentes en train_ml: {missing}"

n_train_ml, n_test_ml = train_ml.count(), test_ml.count()
print(f"Target: {TARGET}")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Columnas excluidas por fuga ({len(LEAKAGE_COLS)}): {sorted(LEAKAGE_COLS)}")
print(f"Filas para modelado tras filtro de duración: train={n_train_ml:,}  test={n_test_ml:,}")

Target: fare_amount
Features (8): ['trip_distance', 'trip_duration_min', 'pu_macro_zone', 'RatecodeID', 'is_flex_fare', 'day_hour_bucket', 'cbd_period_flag', 'passenger_count']
Columnas excluidas por fuga (9): ['Airport_fee', 'cbd_congestion_fee', 'congestion_surcharge', 'extra', 'improvement_surcharge', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount']
Filas para modelado tras filtro de duración: train=797,458  test=199,348


### 4.3 Pipeline de preprocesamiento

El pipeline de `pyspark.ml` encadena tres transformadores antes de cada modelo:

1. **StringIndexer** aprende los índices de las cuatro categóricas sólo en train.
2. **OneHotEncoder** evita que el modelo lineal trate categorías como valores ordinales.
3. **VectorAssembler** concatena las cuatro numéricas (`trip_distance`, `trip_duration_min`, `is_flex_fare`, `passenger_count`) y las categóricas codificadas en `features`.

El mismo pipeline se ajusta en train y luego se aplica a test; categorías nuevas caen bajo `handleInvalid="keep"`.

In [12]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CAT_COLS]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in CAT_COLS],
                       outputCols=[f"{c}_ohe" for c in CAT_COLS])
assembler = VectorAssembler(inputCols=NUM_COLS + [f"{c}_ohe" for c in CAT_COLS],
                           outputCol="features", handleInvalid="keep")

preprocessing_stages = indexers + [encoder, assembler]
print(f"Etapas de preprocesamiento: {len(preprocessing_stages)} ({len(CAT_COLS)} StringIndexers + 1 OneHotEncoder + 1 VectorAssembler)")

Etapas de preprocesamiento: 6 (4 StringIndexers + 1 OneHotEncoder + 1 VectorAssembler)


### 4.4 Modelo baseline: LinearRegression

`LinearRegression` ajusta una combinación lineal de las features (intercepto + suma ponderada) minimizando el error cuadrático medio. Supuestos: la relación entre features y target es razonablemente lineal en cada régimen tarifario (cuestionable globalmente por la discontinuidad del JFK flat fare, pero defendible como baseline). El modelo es interpretable: cada coeficiente da el efecto estimado de un cambio unitario en su feature, manteniendo las demás constantes.

Hiperparámetros: `regParam=0.0` y `elasticNetParam=0.0` (OLS sin regularización), `maxIter=100`. Se usa `regParam=0` como baseline OLS no regularizado; el warning de matriz singular que Spark emite se reporta como limitación del baseline, probablemente asociado a colinealidad o casi colinealidad en la representación categórica. No se optimiza LR porque su función aquí es servir como referencia interpretable frente al bosque aleatorio.

In [13]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import time

lr = LinearRegression(featuresCol="features", labelCol="fare_amount",
                      regParam=0.0, elasticNetParam=0.0, maxIter=100)
pipeline_lr = Pipeline(stages=preprocessing_stages + [lr])

t0 = time.time()
model_lr = pipeline_lr.fit(train_ml)
print(f"LR fit time: {time.time() - t0:.1f}s")

pred_lr = model_lr.transform(test_ml)
ev_rmse = RegressionEvaluator(labelCol="fare_amount", metricName="rmse")
ev_mae  = RegressionEvaluator(labelCol="fare_amount", metricName="mae")
ev_r2   = RegressionEvaluator(labelCol="fare_amount", metricName="r2")
rmse_lr, mae_lr, r2_lr = ev_rmse.evaluate(pred_lr), ev_mae.evaluate(pred_lr), ev_r2.evaluate(pred_lr)
print(f"LR test  RMSE={rmse_lr:.3f}  MAE={mae_lr:.3f}  R2={r2_lr:.4f}")

# Coeficientes con sus nombres
lr_m = model_lr.stages[-1]
feature_names = list(NUM_COLS)
for cat, ix in zip(CAT_COLS, model_lr.stages[:len(CAT_COLS)]):
    feature_names.extend(f"{cat}={lbl}" for lbl in ix.labels)

coefs = lr_m.coefficients.toArray()
print(f"\nIntercept: {lr_m.intercept:.3f} USD")
print(f"Coeficientes ({len(coefs)} features):")
for name, c in zip(feature_names, coefs):
    print(f"  {name:30s} {c:+.4f}")

26/05/24 19:23:18 WARN Instrumentation: [ebe7ba4a] regParam is zero, which might cause numerical instability and overfitting.
26/05/24 19:23:18 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/24 19:23:19 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/05/24 19:23:19 WARN Instrumentation: [ebe7ba4a] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


LR fit time: 4.3s
LR test  RMSE=5.545  MAE=2.481  R2=0.9016

Intercept: 2.415 USD
Coeficientes (21 features):
  trip_distance                  +2.7647
  trip_duration_min              +0.2920
  is_flex_fare                   +9.7699
  passenger_count                +0.1147
  pu_macro_zone=manhattan        -0.3555
  pu_macro_zone=airport          +0.9609
  pu_macro_zone=outer_borough    -0.5995
  pu_macro_zone=unknown          +0.2973
  RatecodeID=1                   +2.6750
  RatecodeID=99                  -6.1268
  RatecodeID=2                   +1.7007
  RatecodeID=5                   +35.4940
  RatecodeID=3                   +27.2260
  RatecodeID=4                   +41.2166
  day_hour_bucket=other          +0.2166
  day_hour_bucket=weekend        -0.0509
  day_hour_bucket=weekday_pm_peak +0.1164
  day_hour_bucket=weekday_am     +0.0014
  day_hour_bucket=late_night     -0.7435
  cbd_period_flag=post_cbd       -0.0639
  cbd_period_flag=pre_cbd        +0.0639


### 4.5 Modelo principal: RandomForestRegressor con tuning

Random Forest entrena un ensemble de árboles de regresión sobre submuestras bootstrap de train y subconjuntos aleatorios de features; la predicción es el promedio de las predicciones individuales. Captura no linealidades e interacciones implícitamente sin requerir feature engineering manual.

Supuestos relajados vs LinearRegression: no asume linealidad ni homoscedasticidad, no es sensible a la escala de features, robusto a outliers moderados. La interpretabilidad se limita a `featureImportances` (reducción promedio de impurity por feature).

Para RF sí aplicamos tuning de hiperparámetros con `TrainValidationSplit` y una grilla compacta:

- `numTrees in [30, 50]`: balancea estabilidad y costo
- `maxDepth in [6, 8]`: brackets del rango usual de profundidad para datos tabulares
- `subsamplingRate in [0.8, 1.0]`: el default de Spark es 1.0 (cada árbol entrena con el 100% del train); bajarlo a 0.8 introduce más decorrelación entre árboles, una técnica clásica de Random Forest que sklearn históricamente usaba por default

Total: 8 combinaciones con `trainRatio=0.75`. El mejor modelo se selecciona por RMSE en la validación interna (25% de train) y se evalúa una sola vez en test al final. Se omite `CrossValidator` porque con 800k filas la varianza entre folds es despreciable y el costo (3x el de TVS) no se justifica.

In [14]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

rf = RandomForestRegressor(featuresCol="features", labelCol="fare_amount",
                           featureSubsetStrategy="auto", seed=42)
pipeline_rf = Pipeline(stages=preprocessing_stages + [rf])

param_grid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [30, 50])
    .addGrid(rf.maxDepth, [6, 8])
    .addGrid(rf.subsamplingRate, [0.8, 1.0])
    .build())

tvs = TrainValidationSplit(estimator=pipeline_rf, estimatorParamMaps=param_grid,
                           evaluator=ev_rmse, trainRatio=0.75, parallelism=2, seed=42)

t0 = time.time()
tvs_model = tvs.fit(train_ml)
print(f"TVS fit time (8 combinaciones): {time.time() - t0:.1f}s")

# Grilla con RMSE de validación interna
print("\nGrid de hiperparámetros (RMSE en validación interna 25%):")
val_rmses = tvs_model.validationMetrics
for params, rmse in zip(param_grid, val_rmses):
    combo = {p.name: v for p, v in params.items()}
    print(f"  {combo} -> RMSE val: {rmse:.4f}")

best_idx = min(range(len(val_rmses)), key=lambda i: val_rmses[i])
best_combo = {p.name: v for p, v in param_grid[best_idx].items()}
print(f"\nMejor configuración: {best_combo}")

# Métricas test con el bestModel (TVS ya refit en train completo)
best_pipeline_rf = tvs_model.bestModel
pred_rf = best_pipeline_rf.transform(test_ml)
rmse_rf, mae_rf, r2_rf = ev_rmse.evaluate(pred_rf), ev_mae.evaluate(pred_rf), ev_r2.evaluate(pred_rf)
print(f"RF (best) test  RMSE={rmse_rf:.3f}  MAE={mae_rf:.3f}  R2={r2_rf:.4f}")

# Importancia de features (todas, en orden de feature)
rf_m = best_pipeline_rf.stages[-1]
fi = rf_m.featureImportances.toArray()
print(f"\nfeatureImportances ({len(fi)} features):")
for name, imp in zip(feature_names, fi):
    print(f"  {name:30s} {imp:.4f}")

26/05/24 19:23:56 WARN DAGScheduler: Broadcasting large task binary with size 1390.0 KiB
26/05/24 19:24:01 WARN DAGScheduler: Broadcasting large task binary with size 1390.0 KiB
26/05/24 19:24:38 WARN DAGScheduler: Broadcasting large task binary with size 1238.2 KiB
26/05/24 19:24:42 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/05/24 19:24:49 WARN DAGScheduler: Broadcasting large task binary with size 1240.1 KiB
26/05/24 19:24:52 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/05/24 19:25:08 WARN DAGScheduler: Broadcasting large task binary with size 1232.7 KiB
26/05/24 19:25:11 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB


TVS fit time (8 combinaciones): 113.4s

Grid de hiperparámetros (RMSE en validación interna 25%):
  {'numTrees': 30, 'maxDepth': 6, 'subsamplingRate': 0.8} -> RMSE val: 5.7087
  {'numTrees': 30, 'maxDepth': 6, 'subsamplingRate': 1.0} -> RMSE val: 5.7281
  {'numTrees': 30, 'maxDepth': 8, 'subsamplingRate': 0.8} -> RMSE val: 5.2450
  {'numTrees': 30, 'maxDepth': 8, 'subsamplingRate': 1.0} -> RMSE val: 5.2864
  {'numTrees': 50, 'maxDepth': 6, 'subsamplingRate': 0.8} -> RMSE val: 5.7635
  {'numTrees': 50, 'maxDepth': 6, 'subsamplingRate': 1.0} -> RMSE val: 5.6439
  {'numTrees': 50, 'maxDepth': 8, 'subsamplingRate': 0.8} -> RMSE val: 5.2973
  {'numTrees': 50, 'maxDepth': 8, 'subsamplingRate': 1.0} -> RMSE val: 5.2295

Mejor configuración: {'numTrees': 50, 'maxDepth': 8, 'subsamplingRate': 1.0}
RF (best) test  RMSE=4.978  MAE=1.751  R2=0.9207

featureImportances (21 features):
  trip_distance                  0.4216
  trip_duration_min              0.2460
  is_flex_fare                   0.0

### 4.6 Comparación y análisis de residuales

La comparación se hace en test usando RMSE, MAE y R². RMSE penaliza errores grandes; MAE es más interpretable en dólares.

También se reporta MAE/RMSE por `is_flex_fare`. Flex Fare usa precio upfront, por lo que se espera peor ajuste que en viajes Metered incluso con distancia y duración observadas.

In [15]:
import pandas as pd

# Tabla comparativa
print("Comparación en test:")
print(pd.DataFrame({
    "Modelo": ["LinearRegression", "RandomForestRegressor (tuned)"],
    "RMSE": [rmse_lr, rmse_rf], "MAE": [mae_lr, mae_rf], "R2": [r2_lr, r2_rf],
}).to_string(index=False, float_format="%.4f"))

# Métricas RF por régimen tarifario (validación empírica del 4.6)
print("\nMétricas RF por régimen tarifario:")
for label, val in [("Metered", 0), ("Flex", 1)]:
    pred_g = pred_rf.filter(F.col("is_flex_fare") == val)
    mae_g  = ev_mae.evaluate(pred_g)
    rmse_g = ev_rmse.evaluate(pred_g)
    print(f"  {label}: MAE={mae_g:.3f}  RMSE={rmse_g:.3f}")

Comparación en test:
                       Modelo   RMSE    MAE     R2
             LinearRegression 5.5449 2.4805 0.9016
RandomForestRegressor (tuned) 4.9780 1.7507 0.9207

Métricas RF por régimen tarifario:
  Metered: MAE=1.241  RMSE=4.221
  Flex: MAE=4.623  RMSE=8.004


## Cierre

**¿El modelo funciona? Sí, condicionado al régimen tarifario.** En viajes Metered, el bosque aleatorio alcanza MAE 1.24 USD, suficiente como primer filtro de auditoría. En Flex Fare el MAE sube a 4.62 USD; el modelo mejora, pero sigue siendo menos confiable para ese régimen. R² agregada de 0.92 indica que las 8 features post-registro capturan buena parte de la estructura tarifaria sin usar componentes de cobro.

**Resultados clave**

El modelo final es un bosque aleatorio (`numTrees=50, maxDepth=8, subsamplingRate=1.0`), seleccionado por `TrainValidationSplit` sobre 8 combinaciones. En test: RMSE 4.98 USD, MAE 1.75 USD, R² 0.92. La regresión lineal baseline alcanza RMSE 5.54 USD, MAE 2.48 USD, R² 0.90.

El RMSE sigue siendo mayor que el MAE, señal de una cola de viajes difíciles (flat rates aeroportuarios, tarifas negociadas y Flex Fare) donde el error excede el promedio absoluto.

Tres lecturas accionables:

1. **La duración cambió materialmente el experimento.** `trip_duration_min` no estaba en la versión inicial del modelo; al añadirla como variable derivada, el bosque aleatorio alcanza MAE 1.75 USD. Esto confirma que la tarifa base depende de distancia y tiempo, no sólo de distancia.

2. **Flex Fare sigue siendo el caso débil.** El MAE queda en 4.62 USD, muy por encima de Metered (1.24 USD). Para uso operativo conviene entrenar modelos separados por régimen.

3. **Distancia y duración dominan.** El bosque atribuye 42% de importancia a `trip_distance` y 25% a `trip_duration_min`; después vienen `pu_macro_zone` y `RatecodeID`. Las variables temporales aportan poco una vez incluida la duración real.

Del tuning, `maxDepth=8` vuelve a ser clave. Con la nueva feature, la mejor configuración usa 50 árboles y `subsamplingRate=1.0`.

**Limitaciones**

- El régimen Flex requiere modelo separado; ya cuantificado arriba.
- La duración aproxima tráfico, pero no separa espera, velocidad ni ruta real.
- La sensibilidad a la semilla aleatoria no se exploró.

**Recomendaciones**

- Validación temporal con datos de 2026.
- Modelos separados Flex vs Metered en producción.

**Declaración de uso de IA**

Google. (2026). *Gemini 3.5 Flash* [Modelo de lenguaje grande], utilizado para el proceso de aprendizaje del contenido de la semana y la validación de errores conceptuales y de código. https://deepmind.google/models/gemini/flash/